# V3.3 — IAM Aachen Ensemble (Scratch Training)

**Accelerator:** GPU T4 x1

**Strateji:** v3_augmented config (84.54% WA) ile 2 model farklı seed eğit,
test'te CTC log-prob ortalaması al (ensemble).

**Değişiklikler vs v3_augmented:**
- N=2 model (seed=42, seed=123), ensemble decode
- Epochs 60→100 (early stopping patience=15, val_loss — orijinal gibi)
- val_loss early stopping korundu (v3.2'deki val_wa deneyi geri alındı)

**Tahmini süre:** ~4 saat (2 model × ~2h early stopping)

**Gerekli datasetslar (Add Data):**
1. `brht25/crnn-h2-code` (scripts + aachen_splits)
2. IAM dataset (`paper-traning-data`)

**Çıktı:** `results/v3_3_results.json`

---
**KURALLAR:** Test set'e sadece bu notebook'ta, bir kez bakılır. Cherry-pick yok.

In [ ]:
# Hücre 1: GPU + donanım bilgisi
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {vram:.0f} GB ({int(vram*1024)} MiB)")
    print(f"PyTorch: {torch.__version__}")

!cat /proc/cpuinfo | grep 'model name' | head -1
!free -h | grep Mem
!python --version

In [ ]:
# Hücre 2: crnn-h2-code dataset'ten kopyala (base), sonra v3.3 override
import sys, os, shutil

CODE_INPUT = "/kaggle/input/datasets/brht25/crnn-h2-code"

if os.path.exists(CODE_INPUT):
    os.makedirs("/kaggle/working/cloud", exist_ok=True)

    # Base scripts (cloud/)
    for fname in os.listdir(f"{CODE_INPUT}/cloud"):
        if fname.endswith((".py", ".txt", ".sh")):
            shutil.copy(f"{CODE_INPUT}/cloud/{fname}", f"/kaggle/working/cloud/{fname}")

    # V3.3: model_v3.py (val_loss early stop) + v3_3_train.py (ensemble)
    v33_src = f"{CODE_INPUT}/cloud/v3.3"
    if os.path.exists(v33_src):
        os.makedirs("/kaggle/working/cloud/v3.3", exist_ok=True)
        for fname in os.listdir(v33_src):
            if fname.endswith((".py", ".txt")):
                shutil.copy(f"{v33_src}/{fname}", f"/kaggle/working/cloud/{fname}")
                shutil.copy(f"{v33_src}/{fname}", f"/kaggle/working/cloud/v3.3/{fname}")
        print("V3.3 override kopyalandı OK")
    else:
        print(f"⚠️  {v33_src} bulunamadı")

    shutil.copytree(f"{CODE_INPUT}/aachen_splits", "/kaggle/working/aachen_splits", dirs_exist_ok=True)
    shutil.copy(f"{CODE_INPUT}/trigram_lm.py", "/kaggle/working/trigram_lm.py")
    print("Scripts + aachen_splits kopyalandı OK")
else:
    print(f"⚠️  {CODE_INPUT} bulunamadı — Add Data → Your Datasets → crnn-h2-code")

sys.path.insert(0, "/kaggle/working")
os.chdir("/kaggle/working")
!pip install -q word-beam-search
!pip install -q -r cloud/requirements.txt

try:
    import word_beam_search
    print("✓ word-beam-search kurulu")
except ImportError:
    print("⚠️  word-beam-search kurulamadı — WBS atlanacak")

In [ ]:
# Hücre 3: IAM dataset path'ini bul
import os, subprocess

print("=== /kaggle/input altındaki tüm datasetler ===")
!ls /kaggle/input/
print()
print("=== words.txt aranıyor (maxdepth 6) ===")
result = subprocess.run(
    ["find", "/kaggle/input", "-name", "words.txt", "-maxdepth", "6"],
    capture_output=True, text=True
)
found_words_txts = [p.strip() for p in result.stdout.strip().splitlines() if p.strip()]
for p in found_words_txts:
    print(" ", p)
if not found_words_txts:
    print("  (bulunamadı — IAM dataset eklendi mi?)")

IAM_WORDS_TXT = None
IAM_WORDS_DIR = None

for wt in found_words_txts:
    candidate_dir = os.path.join(os.path.dirname(wt), "words")
    if os.path.isdir(candidate_dir):
        IAM_WORDS_TXT = wt
        IAM_WORDS_DIR = candidate_dir
        break

if IAM_WORDS_TXT and IAM_WORDS_DIR:
    print(f"\n✓ IAM words.txt : {IAM_WORDS_TXT}")
    print(f"✓ IAM words/    : {IAM_WORDS_DIR}")
else:
    print("\n⚠️  words/ klasörü bulunamadı.")
    for p in found_words_txts:
        print(f"  {p}  →  words/: {os.path.isdir(os.path.join(os.path.dirname(p), 'words'))}")
    raise RuntimeError("IAM dataset bulunamadı — Add Data sekmesinden ekle")

In [ ]:
# Hücre 4: Ensemble eğitimi başlat (~4 saat)
# 2 model × ~2h (early stopping patience=15, val_loss) = ~4h toplam
MODEL_DIR = "/kaggle/working/Model_aachen_v3_3"

!python cloud/v3_3_train.py \
    --epochs 100 \
    --n-ensemble 2 \
    --batch 128 \
    --lr 7e-4 \
    --patience 15 \
    --model-dir {MODEL_DIR} \
    --iam-words {IAM_WORDS_TXT} \
    --iam-root {IAM_WORDS_DIR}

In [ ]:
# Hücre 5: Sonuçları oku ve özetle
import json, os

results_path = "/kaggle/working/results/v3_3_results.json"
if os.path.exists(results_path):
    with open(results_path) as f:
        r = json.load(f)

    print("=" * 55)
    print(" V3.3 ENSEMBLE SONUÇLARI")
    print("=" * 55)
    for j, (s, wa) in enumerate(zip(r['seeds'], r['individual_best_val_wa_pcts'])):
        print(f" Model {j+1} (seed={s}) val WA : {wa:.2f}%")
    print()
    print(f" Ensemble Test WA  : {r['ensemble_wa_pct']:.2f}%")
    ci = r['ensemble_wilson_95ci_pct']
    print(f" Wilson 95% CI     : [{ci[0]:.2f}%, {ci[1]:.2f}%]")
    print(f" Ensemble CER      : {r['ensemble_cer_pct']:.2f}%")
    print(f" N samples         : {r['n_samples']:,}")

    mn = r.get('mcnemar_vs_v3_base', {})
    if mn:
        print(f"\n McNemar (V3-base vs V3.3)")
        print(f" Baseline WA  : {mn.get('baseline_wa_pct', 'N/A'):.2f}%")
        print(f" Delta pp     : {mn.get('delta_pp', 'N/A'):+.2f}pp")
        print(f" p-value      : {mn.get('mcnemar_p', 'N/A'):.2e}")
        print(f" p<.01        : {'YES ✓' if mn.get('significant_p01') else 'NO'}")
    print("=" * 55)
    print("\n--- Arkadaşına gönder ---")
    print(f"Test WA  : {r['ensemble_wa_pct']:.2f}%")
    print(f"Wilson CI: [{ci[0]:.2f}%, {ci[1]:.2f}%]")
    delta = r['ensemble_wa_pct'] - 84.5448
    print(f"\n v3_augmented (84.54%) delta: {delta:+.2f}pp")
else:
    print(f"⚠️ {results_path} bulunamadı — eğitim tamamlandı mı?")